# Transformer (Attention-Based) - TensorFlow / Keras

**Goal:** Classify short text sequences with encoder self-attention.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Self-attention lets each token gather information from every other token.
- **Where it is used:** language models, retrieval, document classification, and multimodal encoders.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Transformer Attention: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['tokens', 'attention', 'context']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = np.exp(-x**2/2)
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.10,.30,.45,.15])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['tok1', 'tok2', 'tok3', 'tok4'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
vocab_size, seq_len, samples = 120, 24, 1600
X = np.random.randint(1, vocab_size, size=(samples, seq_len)).astype("int64")
y = ((X[:, :8].sum(axis=1) + 2 * X[:, 8:16].mean(axis=1)) > 700).astype("int64")
X_train, X_test = X[:1200], X[1200:]
y_train, y_test = y[:1200], y[1200:]


In [ ]:
class TransformerBlock(layers.Layer):
    def __init__(self, d_model=64, heads=4, ff_dim=128):
        super().__init__()
        self.attention = layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads)
        self.ffn = keras.Sequential([layers.Dense(ff_dim, activation="gelu"), layers.Dense(d_model)])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.dropout = layers.Dropout(0.1)

    def call(self, inputs, training=False):
        attended = self.attention(inputs, inputs, training=training)
        x = self.norm1(inputs + self.dropout(attended, training=training))
        feedforward = self.ffn(x)
        return self.norm2(x + self.dropout(feedforward, training=training))


tokens = keras.Input(shape=(seq_len,), dtype="int64")
positions = tf.range(start=0, limit=seq_len, delta=1)
x = layers.Embedding(vocab_size, 64)(tokens) + layers.Embedding(seq_len, 64)(positions)
x = TransformerBlock()(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(2, activation="softmax")(x)
model = keras.Model(tokens, outputs)
model.compile(optimizer=keras.optimizers.AdamW(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])


In [ ]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=8, batch_size=64)